## Importação das bibliotecas
Nessa primeira parte, é feito a importação das bibliotecas, sendo as mais importantes para esse projeto 
- VectorAssembler que é uma função que gera um vetor a partir de um array passado como parâmetro; 
- LinearRegression é um algoritmo de regressão linear do Spark MLlib; 
- RegressionEvaluator é a função que valida a modelo criado.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, year, month, dayofweek, regexp_replace
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
import mlflow
import mlflow.spark
import os

## Definição do volume para alocar o modelo ML

In [0]:
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/previsao_vendas_shoes_dev/source_csv/modelo_ml"

Inicialização do spark

In [0]:
spark = SparkSession.builder.getOrCreate()

Consulta da tabela através de dataframe

In [0]:
df_fato_vendas = spark.read.table("previsao_vendas_shoes_dev.ecommerce.fato_vendas")

## Criação das features e target
Nessa parte nós criamos campos calculados para adicionarmos como features do nosso modelo. Features são as variáveis independentes, e o target é nossa variável dependente.

In [0]:
df_fato_vendas_com_colunas_adicionadas = (
    df_fato_vendas.withColumn("mes", month(col("data_venda")))
    .withColumn("ano", year(col("data_venda")))
    .withColumn("dia_semana", dayofweek(col("data_venda")))
    .withColumn("vlr_pedido", regexp_replace(col("vlr_pedido"), "R\\$|\\s", ""))
    .withColumn("vlr_pedido", regexp_replace(col("vlr_pedido"), ",", ".")) 
    .withColumn("vlr_pedido", col("vlr_pedido").cast("double")) 
    .withColumn("vlr_venda", col("qtd_vendas") * col("vlr_pedido"))
    .withColumn("final_semana", col("dia_semana").isin(6, 7).cast("int"))
      
)
feature_cols = ["ano", "mes", "dia_semana", "qtd_vendas", "vlr_pedido", "final_semana"]
target_col = "vlr_venda"

## Conversão de array numérico em um unico vetor

In [0]:
assemble = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_ml = assemble.transform(df_fato_vendas_com_colunas_adicionadas).select("features", target_col)

## Definição de treinamento e teste do modelo
Nós utilizamos nesse caso a proporção 80% de treino e 20% para teste

In [0]:
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)

## Aplicação do treinamento
Nessa parte realizamos o treinamento com a função LinearRegression e fit, função que treina o modelo usando o conjunto de treino.

In [0]:

lr = LinearRegression(featuresCol="features", labelCol=target_col)
model = lr.fit(train_df)

In [0]:

predictions = model.transform(test_df)
predictions.select("features", target_col, "prediction").show(10)

## Avaliação do modelo com RMSE
RMSE (Root Mean Squared Error = Raiz do erro quadrático médio.), mede quanto, em média, o modelo erra nas previsões.

In [0]:

evaluator = RegressionEvaluator(labelCol=target_col, predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
print(f"RMSE: {rmse}")

## MAE (Mean Absolute Error)
Mede o erro médio absoluto entre previsão e valor real.

In [0]:

mae = RegressionEvaluator(labelCol="vlr_venda", predictionCol="prediction", metricName="mae").evaluate(predictions)
print(f"MAE: {mae}")

## R² (Coeficiente de Determinação)

Mede quanto da variação do target é explicada pelo modelo.

In [0]:

r2 = RegressionEvaluator(labelCol="vlr_venda", predictionCol="prediction", metricName="r2").evaluate(predictions)
print(f"R²: {r2}")

## MLFlow
É uma plataforma para rastrear experimentos (métricas, parâmetros), salvar modelos (com dependências) e Gerenciar versões para deploy.

In [0]:


if mlflow.active_run():
    mlflow.end_run()
else:
    mlflow.start_run()
    mlflow.spark.log_model(model, "modelo_previsao_vendas", pip_requirements=["pyspark==4.0.0", "mlflow"])
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)
    mlflow.log_metric("mae", mae)
    mlflow.end_run()

In [0]:
df_fato_vendas_com_colunas_adicionadas.select("vlr_venda").describe().show()